In [1]:
import pyspark
from pyspark.sql import SparkSession

In [2]:
spark = SparkSession.builder \
    .master("spark://192.168.0.104:7077") \
    .appName('test') \
    .getOrCreate()

In [3]:
spark

In [ ]:
df_green = spark.read.parquet('data/raw/green/*/*')

In [ ]:
df_yellow = spark.read.parquet('data/raw/yellow/*/*')

In [ ]:
df_green.show()

In [ ]:
df_green.printSchema()

In [ ]:
df_yellow.printSchema()

In [ ]:
df_yellow.columns

In [ ]:
df_green.columns

In [ ]:
set(df_green.columns) & set(df_yellow.columns)

In [ ]:
df_green = df_green \
    .withColumnRenamed('lpep_pickup_datetime', 'pickup_datetime') \
    .withColumnRenamed('lpep_dropoff_datetime', 'dropoff_datetime')

In [ ]:
df_yellow = df_yellow \
    .withColumnRenamed('tpep_pickup_datetime', 'pickup_datetime') \
    .withColumnRenamed('tpep_dropoff_datetime', 'dropoff_datetime')

In [ ]:
set(df_green.columns) & set(df_yellow.columns)

In [ ]:
common_colums = []

yellow_columns = set(df_yellow.columns)

for col in df_green.columns:
    if col in yellow_columns:
        common_colums.append(col)

In [ ]:
common_colums

In [ ]:
from pyspark.sql import functions as F

In [ ]:
df_green_sel = df_green \
    .select(common_colums) \
    .withColumn('service_type', F.lit('green'))

In [ ]:
df_yellow_sel = df_yellow \
    .select(common_colums) \
    .withColumn('service_type', F.lit('yellow'))

In [ ]:
df_trips_data = df_green_sel.unionAll(df_yellow_sel)

In [ ]:
df_trips_data.dtypes

In [ ]:
from pyspark.sql.types import *

In [ ]:
target_types = {
    "VendorID": IntegerType(),
    "pickup_datetime": TimestampType(),
    "dropoff_datetime": TimestampType(),
    "store_and_fwd_flag": StringType(),
    "RatecodeID": IntegerType(),
    "PULocationID": IntegerType(),
    "DOLocationID": IntegerType(),
    "passenger_count": IntegerType(),
    "trip_distance": DoubleType(),
    "fare_amount": DoubleType(),
    "extra": DoubleType(),
    "mta_tax": DoubleType(),
    "tip_amount": DoubleType(),
    "tolls_amount": DoubleType(),
    "improvement_surcharge": DoubleType(),
    "total_amount": DoubleType(),
    "payment_type": IntegerType(),
    "congestion_surcharge": DoubleType(),
    "service_type": StringType()
}

In [ ]:
for col_name, col_type in target_types.items():
    
    if col_name in df_trips_data.columns:
        df_trips_data = df_trips_data.withColumn(
            col_name,
            F.col(col_name).cast(col_type)
        )
    else:
        df_trips_data = df_trips_data.withColumn(
            col_name,
            F.lit(None).cast(col_type)
        )

In [ ]:
df_trips_data.dtypes

In [ ]:
df_trips_data.groupBy('service_type').count().show()

In [ ]:
df_trips_data.registerTempTable('trips_data')

In [ ]:
spark.sql("""
SELECT
    service_type,
    count(1)
FROM
    trips_data
GROUP BY 
    service_type
""").show()

In [ ]:
df_result = spark.sql("""
SELECT 
    -- Revenue grouping 
    PULocationID AS revenue_zone,
    date_trunc('month', pickup_datetime) AS revenue_month, 
    service_type, 

    -- Revenue calculation 
    SUM(fare_amount) AS revenue_monthly_fare,
    SUM(extra) AS revenue_monthly_extra,
    SUM(mta_tax) AS revenue_monthly_mta_tax,
    SUM(tip_amount) AS revenue_monthly_tip_amount,
    SUM(tolls_amount) AS revenue_monthly_tolls_amount,
    SUM(improvement_surcharge) AS revenue_monthly_improvement_surcharge,
    SUM(total_amount) AS revenue_monthly_total_amount,
    SUM(congestion_surcharge) AS revenue_monthly_congestion_surcharge,

    -- Additional calculations
    AVG(passenger_count) AS avg_monthly_passenger_count,
    AVG(trip_distance) AS avg_monthly_trip_distance
FROM
    trips_data
GROUP BY
    1, 2, 3
""")

In [ ]:
df_result.show()

In [ ]:
df_result.coalesce(1).write.parquet('data/report/revenue/', mode='overwrite')